### Обоснование стратегий предобработки (ColumnTransformer)

Несмотря на то, что в текущем историческом датасете пропусков нет, пайплайн (Pipeline) должен уметь обрабатывать новые "грязные" данные из продакшена. Поэтому мы закладываем стратегию заполнения пропусков для каждого типа:

1. **Числовые признаки (Sales, Unit price, Quantity, Rating):**
   * **Стратегия заполнения пропусков:** `strategy='median'` (Медиана).
   * **Обоснование:** В прошлой работе на боксплотах мы обнаружили значительные выбросы в правых хвостах (очень крупные покупки). Среднее арифметическое (mean) сильно искажается из-за выбросов, а медиана устойчива к ним.
   * **Масштабирование:** `StandardScaler()`. Необходимо для того, чтобы алгоритмы (например, линейные модели) не отдавали предпочтение признакам с большими значениями (как Sales).

2. **Категориальные признаки (Branch, City, Gender, Product line, Payment):**
   * **Стратегия заполнения пропусков:** `strategy='most_frequent'` (Самая частая категория / мода).
   * **Обоснование:** Для текста мы не можем посчитать среднее. Использование самой частой категории является логичным базовым подходом, не меняющим общую структуру распределения классов.
   * **Кодирование:** `OneHotEncoder(handle_unknown='ignore')`. Переводит текст в нули и единицы. Параметр `ignore` спасает от ошибки, если в тестовой выборке попадется категория, которой не было в тренировочной (предотвращает сбой пайплайна).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Загрузка
df = pd.read_csv('../data/sales.csv')

# 2. Очистка от мусора и зависимых признаков (как решили в EDA)
cols_to_drop = ['Invoice ID', 'Date', 'Time', 'gross margin percentage', 'cogs', 'Tax 5%', 'gross income']
df = df.drop(columns=cols_to_drop)

# 3. Отделяем целевую переменную (будем предсказывать тип клиента)
X = df.drop(columns=['Customer type'])
y = df['Customer type']

# 4. Разделение на train и test ДО преобразований (test откладываем в сторону)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Данные успешно разделены. Утечки нет.")

Данные успешно разделены. Утечки нет.


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Классифицируем оставшиеся признаки
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

# Ветка для числовых признаков
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Ветка для категориальных признаков
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Склеиваем ветки в единый ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

print("ColumnTransformer собран!")

ColumnTransformer собран!


C:\Users\user\AppData\Local\Temp\ipykernel_28488\2842679004.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


In [3]:
# Обучаем препроцессор на X_train и сразу преобразуем
X_train_processed = preprocessor.fit_transform(X_train)

# К X_test применяем ТОЛЬКО transform (без fit!)
X_test_processed = preprocessor.transform(X_test)

# Выводим доказательство правильной работы
print("=== Проверка размерностей ===")
print(f"X_train до обработки:      {X_train.shape}")
print(f"X_train ПОСЛЕ обработки:   {X_train_processed.shape}\n")

print(f"X_test до обработки:       {X_test.shape}")
print(f"X_test ПОСЛЕ обработки:    {X_test_processed.shape}\n")

print("Вывод: Количество строк (800 для train и 200 для test) не изменилось.")
print("Количество колонок увеличилось, так как OneHotEncoder развернул категориальные признаки в новые столбцы.")
print("Утечки данных нет, тест преобразован корректно по правилам трейна.")

=== Проверка размерностей ===
X_train до обработки:      (800, 9)
X_train ПОСЛЕ обработки:   (800, 21)

X_test до обработки:       (200, 9)
X_test ПОСЛЕ обработки:    (200, 21)

Вывод: Количество строк (800 для train и 200 для test) не изменилось.
Количество колонок увеличилось, так как OneHotEncoder развернул категориальные признаки в новые столбцы.
Утечки данных нет, тест преобразован корректно по правилам трейна.
